In [0]:
import json
import re
import time
import uuid
import socket
import ipaddress
from datetime import date, timedelta
from urllib.parse import urlparse

import requests
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType

# ============================================================
# Bronze framework: validates and executes source specs.
# Agents only PROPOSE specs; this deterministic code is the only
# thing that touches the network and the storage.
#   source -> raw files (landing, original format) -> bronze Delta table
# Every HTTP call goes through the API governance (cell 8): budget per quota group, usage log
# and blocks after HTTP 429.
# Requires: _utils_config and _utils_reference_data (%run before this notebook)
# Cells: 1a config/periods/validation · 1b run windows · 1c download/bronze/run · ... · 8 API governance
# ============================================================

bronze_volume_path = f"/Volumes/{catalog}/{schema}/raw"
ingestion_log_table = f"{catalog}.{schema}.bronze_ingestion_log"

allowed_source_types = {"http_api"}          # future: files, jdbc
allowed_methods = {"GET"}
allowed_formats = {"json", "csv"}
allowed_csv_delimiters = {",", ";", "|", "\t"}
allowed_placeholders = {"start", "end"}
allowed_strategies = {"incremental", "full"}
allowed_units = {"day", "month", "year"}
allowed_date_formats = {"%Y-%m-%d", "%Y%m%d", "%d/%m/%Y", "%Y-%m", "%Y%m", "%Y"}
allowed_run_modes = {"auto", "initial", "refresh", "reconcile"}
incremental_only_keys = ["unit", "history_start", "initial_window", "refresh_window", "reconcile_window",
                         "chunk_size", "end_offset", "date_format"]
earliest_history_start = date(2000, 1, 1)
retry_status_codes = {500, 502, 503, 504}    # HTTP 429 is never retried: the quota group is blocked
max_retry_wait_seconds = 120
max_response_bytes = 15 * 1024 * 1024        # VARIANT values are limited to 16 MB
log_batch_size = 10                          # chunks loaded into bronze and logged together

# Limits per unit (windows and chunks are counted in units)
unit_limits = {
    "day":   {"initial_window": 730, "chunk_size": 31, "end_offset": (-10, 2)},
    "month": {"initial_window": 60,  "chunk_size": 12, "end_offset": (-6, 1)},
    "year":  {"initial_window": 10,  "chunk_size": 5,  "end_offset": (-2, 1)},
}

# Fixed calendar anchor (a Monday) for day chunks: with chunk_size = 7 every chunk is a Monday-Sunday week.
# Months always start on day 1 and years on January 1. The same period always falls in the same chunk
# and the same file -> re-runs overwrite, never duplicate.
chunk_anchor = date(2000, 1, 3)

bronze_metadata_columns = ["source_name", "endpoint_name", "source_file", "chunk_start", "chunk_end",
                           "run_id", "ingested_at"]

ingestion_log_schema = StructType([
    StructField("run_id", StringType()),
    StructField("source_name", StringType()),
    StructField("endpoint_name", StringType()),
    StructField("chunk_start", DateType()),
    StructField("chunk_end", DateType()),
    StructField("status", StringType()),
    StructField("http_status", IntegerType()),
    StructField("bytes_written", LongType()),
    StructField("file_path", StringType()),
    StructField("error_message", StringType()),
])


# ------------------------------------------------------------
# Calendar periods (day / month / year)
# ------------------------------------------------------------
def _period_start(d, unit):
    if unit == "day":
        return d
    if unit == "month":
        return d.replace(day=1)
    return d.replace(month=1, day=1)


def _add_periods(period_start, n, unit):
    if unit == "day":
        return period_start + timedelta(days=n)
    if unit == "month":
        months = period_start.year * 12 + (period_start.month - 1) + n
        return date(months // 12, months % 12 + 1, 1)
    return date(period_start.year + n, 1, 1)


def _period_end(period_start, unit):
    return _add_periods(period_start, 1, unit) - timedelta(days=1)


def _period_index(period_start, unit):
    if unit == "day":
        return (period_start - chunk_anchor).days
    if unit == "month":
        return period_start.year * 12 + period_start.month - 1
    return period_start.year


def periods_between(earlier, later, unit):
    """Number of whole periods from the period of 'earlier' to the period of 'later'."""
    return _period_index(_period_start(later, unit), unit) - _period_index(_period_start(earlier, unit), unit)


def _parse_date(value):
    if isinstance(value, date):
        return value
    try:
        return date.fromisoformat(str(value))
    except ValueError:
        return None


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------
def _int_between(value, low, high):
    return isinstance(value, int) and not isinstance(value, bool) and low <= value <= high


def _placeholders(text):
    return set(re.findall(r"\{([^{}]*)\}", str(text)))


def _is_public_host(hostname):
    try:
        for info in socket.getaddrinfo(hostname, 443):
            ip = ipaddress.ip_address(info[4][0])
            if ip.is_private or ip.is_loopback or ip.is_link_local or ip.is_reserved or ip.is_multicast:
                return False
        return True
    except Exception:
        return False


def _validate_locations(locations, params, label):
    """Checks the optional 'locations' block of an endpoint."""
    errors = []
    if not isinstance(locations, dict):
        return [f"endpoint {label}: locations must be a mapping"]

    reference = str(locations.get("reference", ""))
    key_column = locations.get("key_column")
    lat_param = locations.get("latitude_param")
    lon_param = locations.get("longitude_param")

    if not re.fullmatch(r"[A-Za-z0-9_\-]+\.csv", reference):
        errors.append(f"endpoint {label}: locations.reference must be a csv file name in reference_data/ (no paths)")
    for field, value in [("key_column", key_column), ("latitude_param", lat_param), ("longitude_param", lon_param)]:
        if not isinstance(value, str) or not value.strip():
            errors.append(f"endpoint {label}: locations.{field} is required")
    for param in (lat_param, lon_param):
        if isinstance(param, str) and param in params:
            errors.append(f"endpoint {label}: '{param}' is filled from locations; remove it from params")
    if errors:
        return errors

    try:
        rows = read_reference_csv(reference)
    except Exception as e:
        return [f"endpoint {label}: cannot read reference_data/{reference}: {str(e)[:150]}"]
    if not rows:
        return [f"endpoint {label}: reference_data/{reference} is empty"]

    missing = [c for c in (key_column, "latitude", "longitude") if c not in rows[0]]
    if missing:
        return [f"endpoint {label}: reference_data/{reference} has no columns {missing}"]
    keys = [r[key_column] for r in rows]
    if len(keys) != len(set(keys)):
        errors.append(f"endpoint {label}: duplicated values in {reference}.{key_column}")
    for r in rows:
        try:
            float(r["latitude"]), float(r["longitude"])
        except ValueError:
            errors.append(f"endpoint {label}: non-numeric coordinates for {r[key_column]} in {reference}")
            break
    return errors


def _validate_load(load):
    """Checks the load block. Returns (errors, strategy, unit, limits)."""
    errors = []
    if not isinstance(load, dict):
        return ["load is required (strategy: incremental | full)"], None, None, None

    strategy = load.get("strategy")
    if strategy not in allowed_strategies:
        errors.append(f"load.strategy must be one of {sorted(allowed_strategies)}")
        return errors, None, None, None

    if strategy == "full":
        extra = [k for k in incremental_only_keys if k in load]
        if extra:
            errors.append(f"load keys {extra} only apply to the incremental strategy")
        return errors, strategy, None, None

    unit = load.get("unit")
    if unit not in allowed_units:
        errors.append(f"load.unit must be one of {sorted(allowed_units)}")
        return errors, strategy, None, None
    limits = unit_limits[unit]

    # History: exactly one of history_start (fixed date) or initial_window (units back)
    history = None
    has_start, has_window = "history_start" in load, "initial_window" in load
    if has_start == has_window:
        errors.append('load needs exactly one of history_start (a fixed date, e.g. "2025-01-01") '
                      'or initial_window (a number of units back)')
    elif has_start:
        start = _parse_date(load["history_start"])
        if start is None or not (earliest_history_start <= start <= date.today()):
            errors.append(f'load.history_start must be a date between {earliest_history_start} and today, '
                          f'e.g. "2025-01-01"')
        else:
            history = max(1, periods_between(start, date.today(), unit))
    elif not _int_between(load["initial_window"], 1, limits["initial_window"]):
        errors.append(f"load.initial_window must be an integer between 1 and {limits['initial_window']} ({unit}s)")
    else:
        history = load["initial_window"]

    max_window = history or limits["initial_window"]
    refresh_window = load.get("refresh_window")
    if not _int_between(refresh_window, 1, max_window):
        errors.append(f"load.refresh_window must be an integer between 1 and the history length ({max_window} {unit}s): "
                      f"{unit}s re-loaded on every daily run")
    if "reconcile_window" in load:
        low = refresh_window if isinstance(refresh_window, int) else 1
        if not _int_between(load["reconcile_window"], low, max_window):
            errors.append(f"load.reconcile_window must be an integer between refresh_window ({low}) and the "
                          f"history length ({max_window} {unit}s): {unit}s re-loaded by the weekly reconciliation")

    if not _int_between(load.get("chunk_size"), 1, limits["chunk_size"]):
        errors.append(f"load.chunk_size must be an integer between 1 and {limits['chunk_size']} ({unit}s per call)")
    low, high = limits["end_offset"]
    if not _int_between(load.get("end_offset", 0), low, high):
        errors.append(f"load.end_offset must be an integer between {low} and {high} ({unit}s from the current one)")
    if load.get("date_format", "%Y-%m-%d") not in allowed_date_formats:
        errors.append(f"load.date_format must be one of {sorted(allowed_date_formats)}")
    return errors, strategy, unit, limits


def validate_source_spec(spec):
    errors = []
    if not isinstance(spec, dict):
        return ["spec must be a mapping (key: value)"]

    # Source
    if not re.fullmatch(r"[a-z][a-z0-9_]{2,40}", str(spec.get("source_name", ""))):
        errors.append("source_name must be snake_case, 3-41 chars")
    if not str(spec.get("description", "")).strip():
        errors.append("description is required")
    if spec.get("source_type") not in allowed_source_types:
        errors.append(f"source_type must be one of {sorted(allowed_source_types)} (other source types are not supported yet)")

    auth = spec.get("auth", {"type": "none"})
    if not isinstance(auth, dict) or auth.get("type") != "none":
        errors.append("only auth type 'none' is supported for now")

    # Load strategy
    if "incremental" in spec and "load" not in spec:
        errors.append("'incremental' was replaced by 'load' (strategy, unit, history_start or initial_window, "
                      "refresh_window, reconcile_window, chunk_size, end_offset, date_format)")
    load_errors, strategy, unit, limits = _validate_load(spec.get("load"))
    errors += load_errors
    chunk_size = spec["load"].get("chunk_size") if isinstance(spec.get("load"), dict) else None

    # Endpoints
    endpoints = spec.get("endpoints")
    if not isinstance(endpoints, list) or not endpoints:
        errors.append("endpoints must be a non-empty list")
        endpoints = []

    seen_names = set()
    for i, ep in enumerate(endpoints):
        if not isinstance(ep, dict):
            errors.append(f"endpoint #{i} must be a mapping")
            continue
        name = str(ep.get("name", ""))
        label = name or f"#{i}"
        if not re.fullmatch(r"[a-z][a-z0-9_]{1,40}", name):
            errors.append(f"endpoint {label}: name must be snake_case")
        if name in seen_names:
            errors.append(f"endpoint {label}: duplicated name")
        seen_names.add(name)

        if str(ep.get("method", "GET")).upper() not in allowed_methods:
            errors.append(f"endpoint {label}: only GET is allowed")
        if ep.get("response_format") not in allowed_formats:
            errors.append(f"endpoint {label}: response_format must be one of {sorted(allowed_formats)}")
        if "csv_delimiter" in ep:
            if ep.get("response_format") != "csv":
                errors.append(f"endpoint {label}: csv_delimiter is only allowed when response_format is csv")
            elif ep["csv_delimiter"] not in allowed_csv_delimiters:
                errors.append(f"endpoint {label}: csv_delimiter must be one of {sorted(allowed_csv_delimiters)}")

        url = str(ep.get("url", ""))
        parsed = urlparse(url)
        if parsed.scheme != "https" or not parsed.hostname:
            errors.append(f"endpoint {label}: url must be a valid https URL")
        elif not _is_public_host(parsed.hostname):
            errors.append(f"endpoint {label}: host {parsed.hostname} is not a public internet host")
        if parsed.query:
            errors.append(f"endpoint {label}: put query parameters in 'params', not in the url")

        params = ep.get("params", {})
        if not isinstance(params, dict):
            errors.append(f"endpoint {label}: params must be a mapping")
            params = {}
        for key, value in params.items():
            if isinstance(value, bool) or not isinstance(value, (str, int, float)):
                errors.append(f"endpoint {label}: param {key} must be a string or a number")

        used = _placeholders(url).union(*[_placeholders(v) for v in params.values()])
        unknown = used - allowed_placeholders
        if unknown:
            errors.append(f"endpoint {label}: unknown placeholders {sorted(unknown)}; only {{start}} and {{end}} are allowed")
        if strategy == "incremental":
            if "start" not in used:
                errors.append(f"endpoint {label}: incremental endpoints must use {{start}} (in the url or in params)")
            if isinstance(chunk_size, int) and chunk_size > 1 and "end" not in used:
                errors.append(f"endpoint {label}: with chunk_size > 1 the endpoint must also use {{end}}")
        if strategy == "full" and used:
            errors.append(f"endpoint {label}: full strategy endpoints cannot use date placeholders")
        if "end_offset" in ep:
            if limits is None:
                errors.append(f"endpoint {label}: end_offset only applies to the incremental strategy")
            elif not _int_between(ep["end_offset"], *limits["end_offset"]):
                errors.append(f"endpoint {label}: end_offset must be an integer between {limits['end_offset'][0]} "
                              f"and {limits['end_offset'][1]}")

        if "locations" in ep:
            errors += _validate_locations(ep["locations"], params, label)

    # API limits (cell 8)
    errors += validate_limits(spec)
    return errors

In [0]:
# ------------------------------------------------------------
# Which periods does a run cover (decided from the ingestion log, endpoint by endpoint)
# ------------------------------------------------------------
pending_statuses = ("error", "deferred")      # chunks whose latest attempt must be retried


def endpoint_load_state(source_name):
    """Per endpoint: (last chunk_end loaded OK, start of the oldest chunk whose latest attempt failed or was
    deferred). Only rows after the source's last decommission are considered."""
    if not spark.catalog.tableExists(ingestion_log_table):
        return {}
    log = spark.table(ingestion_log_table).filter(F.col("source_name") == source_name)
    decommissioned_at = log.filter(F.col("status") == "decommissioned").agg(F.max("ingested_at")).first()[0]

    attempts = log.filter(F.col("chunk_start").isNotNull() & F.col("status").isin("ok", *pending_statuses))
    if decommissioned_at is not None:
        attempts = attempts.filter(F.col("ingested_at") > F.lit(decommissioned_at))

    latest_attempt = Window.partitionBy("endpoint_name", "chunk_start").orderBy(F.desc("ingested_at"))
    rows = (attempts.withColumn("attempt_rank", F.row_number().over(latest_attempt))
                    .filter(F.col("attempt_rank") == 1)
                    .groupBy("endpoint_name")
                    .agg(F.max(F.when(F.col("status") == "ok", F.col("chunk_end"))).alias("last_ok_end"),
                         F.min(F.when(F.col("status").isin(*pending_statuses), F.col("chunk_start")))
                          .alias("first_failed_start"))
                    .collect())
    return {r["endpoint_name"]: (r["last_ok_end"], r["first_failed_start"]) for r in rows}


def history_window(load, run_date):
    """Length of the full history in units: initial_window, or from history_start to the current period."""
    if "initial_window" in load:
        return load["initial_window"]
    return max(1, periods_between(_parse_date(load["history_start"]), run_date, load["unit"]))


def compute_window(load, endpoint_names, state, run_date, mode):
    """Pure logic (testable without the log). Returns (window_in_units, mode_used)."""
    history = history_window(load, run_date)
    if mode == "initial":
        return history, "initial"
    if mode == "auto" and not any(state.get(e, (None, None))[0] for e in endpoint_names):
        return history, "initial"            # never loaded OK -> first load

    if mode == "reconcile":
        required, mode_used = load.get("reconcile_window", load["refresh_window"]), "reconcile"
    else:
        required, mode_used = load["refresh_window"], "refresh"

    unit = load["unit"]
    for endpoint_name in endpoint_names:
        last_ok_end, first_failed_start = state.get(endpoint_name, (None, None))
        if last_ok_end is None:
            required = max(required, history)                                            # endpoint never loaded
            continue
        required = max(required, periods_between(last_ok_end, run_date, unit) + 1)        # missed runs
        if first_failed_start is not None:
            required = max(required, periods_between(first_failed_start, run_date, unit))  # unresolved failures
    return min(history, required), mode_used


def resolve_window(spec, run_date, mode="auto", window_override=None):
    """How many units back this run covers. Returns (window, mode_used)."""
    if mode not in allowed_run_modes:
        raise ValueError(f"mode must be one of {sorted(allowed_run_modes)}")
    load = spec["load"]
    if load["strategy"] == "full":
        return None, "full"
    if window_override:
        return window_override, "override"
    state = endpoint_load_state(spec["source_name"])
    return compute_window(load, [ep["name"] for ep in spec["endpoints"]], state, run_date, mode)

In [0]:
# ------------------------------------------------------------
# Download helpers (http_api)
# ------------------------------------------------------------
def build_chunks(spec, endpoint, run_date, window=None):
    """Calendar-aligned chunks: the same periods always produce the same chunk boundaries."""
    load = spec["load"]
    if load["strategy"] == "full":
        return [(run_date, run_date)]

    unit, size = load["unit"], load["chunk_size"]
    window = window or history_window(load, run_date)
    end_offset = endpoint.get("end_offset", load.get("end_offset", 0))

    current = _period_start(run_date, unit)
    first = _add_periods(current, -window, unit)
    last_period = _add_periods(current, end_offset, unit)
    last_day = run_date if end_offset == 0 else _period_end(last_period, unit)   # current period: up to today

    # Move first back to the start of its aligned chunk
    chunk_start = _add_periods(first, -(_period_index(first, unit) % size), unit)

    chunks = []
    while chunk_start <= last_day:
        chunk_last_period = _add_periods(chunk_start, size - 1, unit)
        chunks.append((chunk_start, min(_period_end(chunk_last_period, unit), last_day)))   # last chunk may be partial
        chunk_start = _add_periods(chunk_start, size, unit)
    return chunks


def resolve_locations(endpoint):
    """Builds the coordinate params from the reference file. Returns (params, location_codes)."""
    locations = endpoint.get("locations")
    if not locations:
        return {}, None
    rows = read_reference_csv(locations["reference"])
    location_params = {
        locations["latitude_param"]: ",".join(r["latitude"] for r in rows),
        locations["longitude_param"]: ",".join(r["longitude"] for r in rows),
    }
    return location_params, [r[locations["key_column"]] for r in rows]


def _render(value, start_str, end_str):
    if isinstance(value, str):
        return value.replace("{start}", start_str).replace("{end}", end_str)
    return value


def _http_get(url, params, retries=3):
    """GET with retries for 5xx and network errors (back-off 1, 2 s).
    HTTP 429 is NEVER retried: the caller blocks the quota group (see cell 8).
    Returns status_code, body, error, attempts (each attempt counts as a call) and retry_after."""
    result = {"status_code": None, "body": None, "error": "not executed", "attempts": 0, "retry_after": None}
    for attempt in range(retries):
        result["attempts"] = attempt + 1
        try:
            response = requests.get(
                url, params=params, timeout=60, allow_redirects=False,
                headers={"User-Agent": "databricks-bronze-framework/1.0"},
            )
            result.update(status_code=response.status_code, body=response.content, error=None)
            if response.status_code == 200:
                return result
            result["error"] = f"HTTP {response.status_code}: {response.text[:300]}"
            if response.status_code == 429:
                result["retry_after"] = response.headers.get("Retry-After")
                return result
            if response.status_code not in retry_status_codes:
                return result
        except requests.RequestException as e:
            result.update(status_code=None, body=None, error=str(e)[:300])
        if attempt < retries - 1:
            time.sleep(min(2 ** attempt, max_retry_wait_seconds))
    return result


def _check_body(body, response_format):
    if not body:
        return "empty response"
    if len(body) > max_response_bytes:
        return f"response larger than {max_response_bytes} bytes"
    text = body.decode("utf-8", errors="replace")
    if response_format == "json":
        try:
            json.loads(text)
        except ValueError as e:
            return f"invalid JSON: {str(e)[:200]}"
    if response_format == "csv" and len(text.strip().splitlines()) < 2:
        return "CSV has no data rows"
    return None


def _top_level_items(text, response_format):
    """Number of elements when the response is a JSON list (e.g. one per location)."""
    if response_format != "json":
        return None
    try:
        parsed = json.loads(text)
    except ValueError:
        return None
    return len(parsed) if isinstance(parsed, list) else 1


# ------------------------------------------------------------
# Bronze Delta tables (one per endpoint)
# ------------------------------------------------------------
def bronze_table_name(source_name, endpoint_name):
    return f"{catalog}.{schema}.{source_name}_{endpoint_name}"


def _safe_column_name(name, used):
    base = re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_") or "col"
    if base[0].isdigit():
        base = f"c_{base}"
    candidate, i = base, 2
    while candidate in used:
        candidate = f"{base}_{i}"
        i += 1
    used.add(candidate)
    return candidate


def _file_path_col():
    # _metadata.file_path may come as dbfs:/Volumes/... -> normalize to /Volumes/...
    return F.regexp_replace(F.col("_metadata.file_path"), r"^[a-z]+:", "").alias("source_file")


def load_bronze_tables(spec, results, run_id):
    """Load the given files into bronze Delta tables, replacing previous rows of the same files."""
    loaded = {}
    for ep in spec["endpoints"]:
        files = [r for r in results if r["endpoint_name"] == ep["name"] and r["status"] == "ok"]
        if not files:
            continue

        table = bronze_table_name(spec["source_name"], ep["name"])
        paths = [r["file_path"] for r in files]
        has_locations = "locations" in ep

        if has_locations:
            file_info = spark.createDataFrame(
                [(r["file_path"], r["chunk_start"], r["chunk_end"], r["location_codes"]) for r in files],
                "source_file STRING, chunk_start DATE, chunk_end DATE, location_codes ARRAY<STRING>",
            )
        else:
            file_info = spark.createDataFrame(
                [(r["file_path"], r["chunk_start"], r["chunk_end"]) for r in files],
                "source_file STRING, chunk_start DATE, chunk_end DATE",
            )

        if ep["response_format"] == "json":
            # One row per file, whole document stored as VARIANT
            df = (spark.read.option("wholetext", True).text(paths)
                  .select(F.expr("parse_json(value)").alias("payload"), _file_path_col()))
        else:
            # One row per CSV line, every column as STRING (typing is silver's job)
            raw = (spark.read.option("header", True).option("inferSchema", False)
                   .option("sep", ep.get("csv_delimiter", ",")).csv(paths))
            used = set(bronze_metadata_columns) | {"location_codes"}
            df = raw.select(
                *[F.col(f"`{c}`").cast("string").alias(_safe_column_name(c, used)) for c in raw.columns],
                _file_path_col(),
            )

        df = (df.join(F.broadcast(file_info), "source_file", "left")
                .withColumn("source_name", F.lit(spec["source_name"]))
                .withColumn("endpoint_name", F.lit(ep["name"]))
                .withColumn("run_id", F.lit(run_id))
                .withColumn("ingested_at", F.current_timestamp()))

        if df.filter(F.col("chunk_start").isNull()).limit(1).count() > 0:
            raise RuntimeError(f"{table}: could not match some rows to their source file")

        extra_metadata = ["location_codes"] if has_locations else []
        data_columns = [c for c in df.columns if c not in bronze_metadata_columns + extra_metadata]
        df = df.select(*bronze_metadata_columns, *extra_metadata, *data_columns)

        if spark.catalog.tableExists(table):
            files_sql = ", ".join(f"'{p}'" for p in paths)
            (df.write.mode("overwrite")
               .option("replaceWhere", f"source_file IN ({files_sql})")   # replace rows of these files only
               .option("mergeSchema", "true")
               .saveAsTable(table))
        else:
            df.write.saveAsTable(table)

        loaded[table] = len(paths)
        print(f"  ↳ bronze table {table}: {len(paths)} file(s) loaded")
    return loaded


# ------------------------------------------------------------
# Run a spec: budget check -> download -> raw files -> (every log_batch_size chunks) bronze tables, then log
# ------------------------------------------------------------
def _flush_batch(spec, batch, run_id):
    """Loads a batch of chunks into bronze and THEN logs them, so a chunk logged as 'ok' is always in bronze.
    If the run stops before this point, those chunks are not logged and the next run downloads them again."""
    if not batch:
        return {}
    loaded = {}
    if any(r["status"] == "ok" for r in batch):
        try:
            loaded = load_bronze_tables(spec, batch, run_id)
        except Exception as e:
            raise RuntimeError(f"Loading bronze failed for {len(batch)} chunk(s) (run_id {run_id}). The raw files "
                               f"were written but not logged, so the next run will download them again: {e}")
    log_rows = [(run_id, spec["source_name"], r["endpoint_name"], r["chunk_start"], r["chunk_end"], r["status"],
                 r["http_status"], r["bytes_written"], r["file_path"], r["error_message"]) for r in batch]
    (spark.createDataFrame(log_rows, ingestion_log_schema)
          .withColumn("ingested_at", F.current_timestamp())
          .write.mode("append").saveAsTable(ingestion_log_table))
    return loaded


def wait_for_budget(recent_calls, limits, cost):
    """Budget decision for one call, waiting (up to max_budget_wait_seconds) when only the per-minute
    budget or the minimum gap is exceeded. Returns ('call' | 'defer' | 'too_big', reason)."""
    waited = 0.0
    decision, detail = budget_decision(recent_calls, limits, cost, utc_now())
    while decision == "wait" and waited < max_budget_wait_seconds:
        time.sleep(detail)
        waited += detail
        decision, detail = budget_decision(recent_calls, limits, cost, utc_now())
    if decision == "wait":
        return "defer", f"API budget: waited {waited:.0f} s without room"
    return decision, detail


def run_source_spec(spec, window_override=None, dry_run=False, run_date=None, mode="auto", caller=None):
    """mode: auto (first load, then daily refresh), refresh, reconcile (weekly) or initial (full history).
    window_override: number of units back, for tests and dry runs. caller: run | dry_run | ... (usage log).
    Chunks are processed from the most recent to the oldest; when the API budget runs out (or the quota group
    is blocked) the remaining chunks are 'deferred' and retried by later runs."""
    errors = validate_source_spec(spec)
    if errors:
        raise ValueError(f"Invalid source spec: {errors}")

    run_date = run_date or date.today()
    run_id = str(uuid.uuid4())
    source_name = spec["source_name"]
    load = spec["load"]
    date_format = load.get("date_format", "%Y-%m-%d")
    window, mode_used = resolve_window(spec, run_date, mode, window_override)
    results, batch, bronze_tables = [], [], {}

    limits = source_limits(spec)
    quota_group = limits["quota_group"]
    caller = caller or ("dry_run" if dry_run else "run")
    create_api_governance_tables()
    recent_calls = load_api_usage(quota_group)
    usage_rows = []

    stop = None                                   # (status, reason) applied to every remaining chunk
    block = active_block(quota_group)
    if block:
        stop = ("deferred", f"quota group {quota_group} blocked until {block['blocked_until']} UTC "
                            f"({str(block['reason'])[:150]})")

    print(f"▶ {source_name}: {mode_used}"
          + (f" · {window} {load['unit']}(s) back" if window else "")
          + (" · dry run" if dry_run else "")
          + f" · quota group {quota_group}"
          + (" · BLOCKED" if block else ""))

    if not dry_run:
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw")

    endpoints = {}
    for ep in spec["endpoints"]:
        folder = f"{bronze_volume_path}/{source_name}/{ep['name']}"
        location_params, location_codes = resolve_locations(ep)
        if not dry_run:
            dbutils.fs.mkdirs(folder)
        endpoints[ep["name"]] = (ep, folder, location_params, location_codes)

    # Most recent chunks first: if the budget runs out, what is left is history (completed by later runs)
    work = sorted(((chunk_start, chunk_end, ep["name"]) for ep in spec["endpoints"]
                   for chunk_start, chunk_end in build_chunks(spec, ep, run_date, window)), reverse=True)

    try:
        for chunk_start, chunk_end, endpoint_name in work:
            ep, folder, location_params, location_codes = endpoints[endpoint_name]
            fmt = ep["response_format"]
            file_path, bytes_written, sample, items, http_status = None, 0, None, None, None

            start_str, end_str = chunk_start.strftime(date_format), chunk_end.strftime(date_format)
            url = _render(ep["url"], start_str, end_str)
            params = {k: _render(v, start_str, end_str) for k, v in ep.get("params", {}).items()}
            params.update(location_params)
            cost = estimate_call_cost(limits, params, chunk_start, chunk_end, len(location_codes or []))

            if stop is None:
                decision, detail = wait_for_budget(recent_calls, limits, cost)
                if decision == "defer":
                    stop = ("deferred", detail)
                elif decision == "too_big":
                    stop = ("error", detail)

            if stop:
                status, error = stop[0], f"skipped: {stop[1]}"
            else:
                response = _http_get(url, params)
                called_at = utc_now()
                spent = cost * max(1, response["attempts"])
                recent_calls.append((called_at, spent))
                usage_rows.append((quota_group, source_name, endpoint_name, caller, run_id, called_at,
                                   float(spent), response["status_code"]))
                http_status = response["status_code"]
                error = response["error"] or _check_body(response["body"], fmt)

                if http_status == 429:
                    until = block_until_from_retry_after(response["retry_after"])
                    block_quota_group(quota_group, until, f"HTTP 429 from {source_name}.{endpoint_name}: {error}")
                    stop = ("deferred", f"quota group {quota_group} blocked until {until} UTC after HTTP 429")

                if not error:
                    text = response["body"].decode("utf-8", errors="replace")
                    sample = text[:1500]
                    items = _top_level_items(text, fmt)
                    if not dry_run:
                        # File name = aligned chunk start -> same chunk, same file, always overwritten
                        file_path = f"{folder}/{ep['name']}_{chunk_start}.{fmt}"
                        dbutils.fs.put(file_path, text, overwrite=True)
                        bytes_written = len(response["body"])
                status = "error" if error else ("dry_run_ok" if dry_run else "ok")

            print(f"{source_name}.{endpoint_name} {chunk_start}->{chunk_end}: {status}"
                  + (f" (cost {cost:g})" if status in ("ok", "dry_run_ok") else "")
                  + (f" ({error[:120]})" if error else ""))
            result = {
                "endpoint_name": endpoint_name, "chunk_start": chunk_start, "chunk_end": chunk_end,
                "status": status, "http_status": http_status, "bytes_written": bytes_written,
                "file_path": file_path, "error_message": error, "sample": sample,
                "top_level_items": items, "location_codes": location_codes,
            }
            results.append(result)

            if not dry_run:
                batch.append(result)
                if len(batch) >= log_batch_size:
                    for table, n in _flush_batch(spec, batch, run_id).items():
                        bronze_tables[table] = bronze_tables.get(table, 0) + n
                    batch = []
                    record_api_usage(usage_rows)
                    usage_rows = []
    finally:
        record_api_usage(usage_rows)              # usage is recorded even if the run fails
        usage_rows = []

    if not dry_run:
        for table, n in _flush_batch(spec, batch, run_id).items():
            bronze_tables[table] = bronze_tables.get(table, 0) + n

    return {
        "run_id": run_id,
        "source_name": source_name,
        "quota_group": quota_group,
        "mode": mode_used,
        "window": window,
        "unit": load.get("unit"),
        "chunks_ok": sum(1 for r in results if r["status"] in ("ok", "dry_run_ok")),
        "chunks_error": sum(1 for r in results if r["status"] == "error"),
        "chunks_deferred": sum(1 for r in results if r["status"] == "deferred"),
        "bronze_tables": bronze_tables,
        "results": results,
    }

In [0]:
import yaml
from datetime import date as _date

# ------------------------------------------------------------
# YAML support: humans read/write YAML, the framework works with dicts
# ------------------------------------------------------------
def load_source_spec(yaml_text):
    """Parse a YAML source spec into a dict (dates are normalized to ISO strings)."""
    try:
        spec = yaml.safe_load(yaml_text)
    except yaml.YAMLError as e:
        raise ValueError(f"Invalid YAML: {str(e)[:300]}")
    if not isinstance(spec, dict):
        raise ValueError("YAML spec must be a mapping (key: value) at the top level")
    load = spec.get("load")
    if isinstance(load, dict) and isinstance(load.get("history_start"), _date):
        load["history_start"] = load["history_start"].isoformat()   # unquoted dates become date objects in YAML
    return spec


def spec_to_yaml(spec):
    """Render a spec dict as readable YAML (for review and approval)."""
    return yaml.safe_dump(spec, sort_keys=False, allow_unicode=True, default_flow_style=False)

In [0]:
import os

# ------------------------------------------------------------
# 5. Spec files: one YAML file per source in the source_specs/ folder (versioned in Git)
# ------------------------------------------------------------
def find_source_specs_dir(folder_name="source_specs"):
    """Walk up from the running notebook's folder until source_specs/ is found."""
    current = os.getcwd()
    while True:
        candidate = os.path.join(current, folder_name)
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(f"Folder '{folder_name}' not found from {os.getcwd()} upwards")
        current = parent


def list_source_specs():
    return sorted(f[:-4] for f in os.listdir(find_source_specs_dir()) if f.endswith(".yml"))


def load_source_spec_file(source_name):
    path = os.path.join(find_source_specs_dir(), f"{source_name}.yml")
    with open(path, encoding="utf-8") as f:
        spec = load_source_spec(f.read())
    if spec.get("source_name") != source_name:
        raise ValueError(f"{path}: source_name '{spec.get('source_name')}' must match the file name")
    return spec


def save_source_spec_file(spec):
    """Validate and write a spec to source_specs/<source_name>.yml (used by the agent after approval)."""
    errors = validate_source_spec(spec)
    if errors:
        raise ValueError(f"Invalid source spec, not saved: {errors}")
    path = os.path.join(find_source_specs_dir(), f"{spec['source_name']}.yml")
    with open(path, "w", encoding="utf-8") as f:
        f.write(spec_to_yaml(spec))
    return path

In [0]:
# ------------------------------------------------------------
# 6. probe_url: safe exploratory GET for agents (never stores anything)
# ------------------------------------------------------------
max_probe_bytes = 2 * 1024 * 1024
max_probes_per_session = 20
probe_calls = {"count": 0}


def reset_probe_budget():
    probe_calls["count"] = 0


def _first_informative(items):
    """First list element that is not null/empty, to describe the list."""
    for item in items:
        if item not in (None, "", [], {}):
            return item
    return items[0]


def _json_outline(value, depth=0, max_depth=10, max_keys=25):
    """Compact structure of a JSON document, with example values, for an LLM to read."""
    if depth >= max_depth:
        return "..."
    if isinstance(value, dict):
        outline = {k: _json_outline(v, depth + 1) for k, v in list(value.items())[:max_keys]}
        if len(value) > max_keys:
            outline["..."] = f"{len(value) - max_keys} more keys"
        return outline
    if isinstance(value, list):
        if not value:
            return "array[0]"
        return {f"array[{len(value)}] of": _json_outline(_first_informative(value), depth + 1)}
    if isinstance(value, bool):
        return f"bool (e.g. {value})"
    if isinstance(value, (int, float)):
        return f"number (e.g. {value})"
    if value is None:
        return "null"
    return f"string (e.g. {str(value)[:40]!r})"


def _detect_format(text, content_type):
    stripped = text.lstrip()
    content_type = content_type.lower()
    if "json" in content_type or stripped[:1] in ("{", "["):
        try:
            json.loads(text)
            return "json"
        except ValueError:
            pass
    if "html" in content_type or "xml" in content_type or stripped[:1] == "<":
        return "html_or_xml"
    lines = text.splitlines()
    if len(lines) >= 2 and any(d in lines[0] for d in allowed_csv_delimiters):
        return "csv"
    return "text"


def probe_url(url, params=None):
    """Exploratory GET with the same safety rules as the framework. Returns a compact description."""
    params = params or {}
    errors = []
    parsed = urlparse(str(url))
    if parsed.scheme != "https" or not parsed.hostname:
        errors.append("url must be a valid https URL")
    elif not _is_public_host(parsed.hostname):
        errors.append(f"host {parsed.hostname} is not a public internet host")
    if parsed.query:
        errors.append("put query parameters in 'params', not in the url")
    if not isinstance(params, dict):
        errors.append("params must be an object")
        params = {}
    for key, value in params.items():
        if isinstance(value, bool) or not isinstance(value, (str, int, float)):
            errors.append(f"param {key} must be a string or a number")
    if errors:
        return {"status": "rejected", "errors": errors}

    if probe_calls["count"] >= max_probes_per_session:
        return {"status": "rejected",
                "errors": [f"probe budget exhausted ({max_probes_per_session} calls); "
                           "propose a spec with what you know, or ask the user"]}
    probe_calls["count"] += 1

    try:
        response = requests.get(
            url, params=params, timeout=30, allow_redirects=False, stream=True,
            headers={"User-Agent": "databricks-bronze-framework/1.0"},
        )
        body = response.raw.read(max_probe_bytes + 1, decode_content=True)
        response.close()
    except requests.RequestException as e:
        return {"status": "error", "error": str(e)[:300]}

    result = {
        "status": "ok" if response.status_code == 200 else "http_error",
        "http_status": response.status_code,
        "requested_url": response.url,
        "content_type": response.headers.get("Content-Type", ""),
        "bytes_read": min(len(body), max_probe_bytes),
        "truncated": len(body) > max_probe_bytes,
        "probes_left": max_probes_per_session - probe_calls["count"],
    }

    if 300 <= response.status_code < 400:
        result["redirect_to"] = response.headers.get("Location")
        result["hint"] = "Redirects are not followed. Probe redirect_to directly if you trust it (it will be validated)."
        return result

    text = body[:max_probe_bytes].decode("utf-8", errors="replace")
    if result["truncated"]:
        result["hint"] = "Response is larger than the probe limit: narrow the date range or filters."
    else:
        detected = _detect_format(text, result["content_type"])
        result["detected_format"] = detected
        if detected == "json":
            result["structure"] = _json_outline(json.loads(text))
        elif detected == "csv":
            lines = text.splitlines()
            result["csv_delimiter_guess"] = max(allowed_csv_delimiters, key=lambda d: lines[0].count(d))
            result["csv_preview"] = lines[:6]
    result["sample"] = text[:1500]
    return result

In [0]:
import os

# ------------------------------------------------------------
# 7. Decommission a source (admin operation, never exposed to agents)
# ------------------------------------------------------------
def describe_source_footprint(source_name):
    """Everything that exists in the platform for a source."""
    spec = load_source_spec_file(source_name)
    raw_folder = f"{bronze_volume_path}/{source_name}"

    raw_files = 0
    for ep in spec["endpoints"]:
        try:
            raw_files += len(dbutils.fs.ls(f"{raw_folder}/{ep['name']}"))
        except Exception:
            pass   # folder does not exist

    tables = [bronze_table_name(source_name, ep["name"]) for ep in spec["endpoints"]]
    log_rows = (spark.table(ingestion_log_table).filter(F.col("source_name") == source_name).count()
                if spark.catalog.tableExists(ingestion_log_table) else 0)

    return {
        "source_name": source_name,
        "endpoints": [ep["name"] for ep in spec["endpoints"]],
        "spec_path": os.path.join(find_source_specs_dir(), f"{source_name}.yml"),
        "raw_folder": raw_folder,
        "raw_files": raw_files,
        "bronze_tables": [t for t in tables if spark.catalog.tableExists(t)],
        "log_rows": log_rows,
    }


def decommission_source(source_name):
    """Removes a source: bronze tables, raw files and spec (in that order). The log history is kept."""
    if source_name not in list_source_specs():
        raise ValueError(f"Source '{source_name}' has no spec in source_specs/; nothing to decommission")

    footprint = describe_source_footprint(source_name)
    print(f"\n===== DECOMMISSION SOURCE: {source_name} =====")
    print(f"Spec file     : {footprint['spec_path']}")
    print(f"Raw files     : {footprint['raw_files']} file(s) in {footprint['raw_folder']}")
    print(f"Bronze tables : {footprint['bronze_tables'] or 'none'}")
    print(f"Log rows      : {footprint['log_rows']} (kept: ingestion history is never deleted)")

    answer = input(f"\nType the source name to confirm ({source_name}): ").strip()
    if answer != source_name:
        print("❌ Cancelled: nothing was deleted")
        return {"status": "cancelled", "source_name": source_name}

    # 1. Bronze tables
    for table in footprint["bronze_tables"]:
        spark.sql(f"DROP TABLE IF EXISTS {table}")
        print(f"🗑️ Dropped table {table}")

    # 2. Raw files
    try:
        dbutils.fs.rm(footprint["raw_folder"], True)
        print(f"🗑️ Removed raw folder {footprint['raw_folder']}")
    except Exception as e:
        print(f"⚠️ Raw folder not removed ({str(e)[:100]})")

    # 3. Spec file, last: if anything above failed, the spec is still there to retry
    os.remove(footprint["spec_path"])
    print(f"🗑️ Removed spec {footprint['spec_path']}")

    # 4. Leave a trace in the ingestion log
    run_id = str(uuid.uuid4())
    log_rows = [(run_id, source_name, ep, None, None, "decommissioned", None, 0, footprint["raw_folder"],
                 "source decommissioned: spec, raw files and bronze tables removed")
                for ep in footprint["endpoints"]]
    (spark.createDataFrame(log_rows, ingestion_log_schema)
          .withColumn("ingested_at", F.current_timestamp())
          .write.mode("append").saveAsTable(ingestion_log_table))

    print(f"\n✅ Source {source_name} decommissioned · commit the deleted spec in Git to complete the review")
    return {"status": "decommissioned", "source_name": source_name, "run_id": run_id}

In [0]:
import math
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

# ============================================================
# 8. API governance: never overuse a data source.
#   - limits per spec (optional; prudent defaults otherwise): per_minute / per_hour / per_day,
#     min_seconds_between_calls, cost of a call, quota_group (sources sharing the same provider quota)
#   - every HTTP call (runs, dry runs, probes) is recorded in bronze_api_usage with its estimated cost
#   - before a call: usage of the quota group on sliding windows vs limit x api_safety_margin
#       per minute / min gap -> wait · per hour / per day -> defer (retried by later runs)
#   - after an HTTP 429: the quota group is blocked (bronze_source_blocks) until Retry-After or the next
#     UTC midnight, for every caller (job, manual runs, agents)
# Timestamps are UTC (the session time zone is set to UTC here).
# ============================================================
api_usage_table = f"{catalog}.{schema}.bronze_api_usage"
source_blocks_table = f"{catalog}.{schema}.bronze_source_blocks"
api_safety_margin = 0.8
default_limits = {"per_minute": 60, "per_hour": 1000, "per_day": 5000, "min_seconds_between_calls": 1.0}
allowed_limit_keys = {"quota_group", "per_minute", "per_hour", "per_day", "min_seconds_between_calls", "cost",
                      "terms_url"}
allowed_cost_keys = {"per_location", "days_per_unit", "variables_param", "variables_per_unit"}
budget_windows = [("per_minute", 60), ("per_hour", 3600), ("per_day", 86400)]
max_budget_wait_seconds = 90


def utc_now():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def validate_limits(spec):
    """Checks the optional limits block of a spec."""
    limits = spec.get("limits")
    if limits is None:
        return []
    if not isinstance(limits, dict):
        return ["limits must be a mapping"]
    errors = []
    unknown = set(limits) - allowed_limit_keys
    if unknown:
        errors.append(f"limits: unknown keys {sorted(unknown)}")
    if "quota_group" in limits and not re.fullmatch(r"[a-z][a-z0-9_]{2,40}", str(limits["quota_group"])):
        errors.append("limits.quota_group must be snake_case (sources that share a provider quota use the same group)")
    for key in ("per_minute", "per_hour", "per_day"):
        if key in limits and not _int_between(limits[key], 1, 10_000_000):
            errors.append(f"limits.{key} must be a positive integer (calls allowed by the provider)")
    merged = {**default_limits, **{k: v for k, v in limits.items() if k in default_limits}}
    if all(isinstance(merged[k], int) for k in ("per_minute", "per_hour", "per_day")) and \
            not merged["per_minute"] <= merged["per_hour"] <= merged["per_day"]:
        errors.append("limits must satisfy per_minute <= per_hour <= per_day")
    gap = limits.get("min_seconds_between_calls", 1.0)
    if isinstance(gap, bool) or not isinstance(gap, (int, float)) or not 0 <= gap <= 60:
        errors.append("limits.min_seconds_between_calls must be a number between 0 and 60")

    cost = limits.get("cost", {})
    if not isinstance(cost, dict):
        errors.append("limits.cost must be a mapping")
        cost = {}
    unknown = set(cost) - allowed_cost_keys
    if unknown:
        errors.append(f"limits.cost: unknown keys {sorted(unknown)}")
    endpoints = [ep for ep in spec.get("endpoints") or [] if isinstance(ep, dict)]
    if "per_location" in cost:
        if not isinstance(cost["per_location"], bool):
            errors.append("limits.cost.per_location must be true or false")
        elif cost["per_location"] and not any("locations" in ep for ep in endpoints):
            errors.append("limits.cost.per_location needs an endpoint with a locations block")
    for key, high in (("days_per_unit", 366), ("variables_per_unit", 1000)):
        if key in cost and not _int_between(cost[key], 1, high):
            errors.append(f"limits.cost.{key} must be an integer between 1 and {high}")
    if "variables_param" in cost:
        if not any(cost["variables_param"] in (ep.get("params") or {}) for ep in endpoints):
            errors.append(f"limits.cost.variables_param '{cost['variables_param']}' is not a param of any endpoint")
        if "variables_per_unit" not in cost:
            errors.append("limits.cost.variables_param needs variables_per_unit")
    if "terms_url" in limits and urlparse(str(limits["terms_url"])).scheme != "https":
        errors.append("limits.terms_url must be an https URL")
    return errors


def source_limits(spec):
    """Effective limits of a spec: declared values over the defaults; quota_group defaults to the source."""
    limits = {**default_limits, "quota_group": spec["source_name"], "cost": {}}
    limits.update(spec.get("limits") or {})
    return limits


def estimate_call_cost(limits, params, chunk_start, chunk_end, location_count):
    """Calls the provider counts for one request (e.g. Open-Meteo: per location, per 14 days, per 10 variables)."""
    rules = limits.get("cost") or {}
    cost = 1.0
    if rules.get("per_location") and location_count:
        cost *= location_count
    if rules.get("days_per_unit") and chunk_start and chunk_end:
        cost *= max(1.0, ((chunk_end - chunk_start).days + 1) / rules["days_per_unit"])
    if rules.get("variables_param") and rules.get("variables_per_unit"):
        variables = [v for v in str(params.get(rules["variables_param"], "")).split(",") if v.strip()]
        cost *= max(1.0, len(variables) / rules["variables_per_unit"])
    return round(cost, 2)


def create_api_governance_tables():
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {api_usage_table} (
        quota_group STRING, source_name STRING, endpoint_name STRING, caller STRING, run_id STRING,
        called_at TIMESTAMP, cost DOUBLE, http_status INT)""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {source_blocks_table} (
        quota_group STRING, blocked_at TIMESTAMP, blocked_until TIMESTAMP, reason STRING, unblocked_at TIMESTAMP)""")


def load_api_usage(quota_group):
    """Calls of the last 24 h of a quota group: [(called_at, cost)]."""
    since = utc_now() - timedelta(days=1)
    rows = (spark.table(api_usage_table)
            .filter((F.col("quota_group") == quota_group) & (F.col("called_at") >= F.lit(since)))
            .select("called_at", "cost").collect())
    return [(r["called_at"], r["cost"]) for r in rows]


def record_api_usage(rows):
    """rows: (quota_group, source_name, endpoint_name, caller, run_id, called_at, cost, http_status)."""
    if rows:
        schema_ddl = ("quota_group STRING, source_name STRING, endpoint_name STRING, caller STRING, run_id STRING, "
                      "called_at TIMESTAMP, cost DOUBLE, http_status INT")
        spark.createDataFrame(rows, schema_ddl).write.mode("append").saveAsTable(api_usage_table)


def budget_decision(calls, limits, cost, now):
    """Pure logic. calls: [(called_at, cost)] of the quota group.
    Returns ('call', None) | ('wait', seconds) | ('defer', reason) | ('too_big', reason)."""
    for name, seconds in budget_windows:
        allowed = limits[name] * api_safety_margin
        if cost > allowed:
            return "too_big", (f"one call costs {cost:g}, more than the {name} budget ({allowed:g}); "
                               f"reduce chunk_size or the number of locations")
        in_window = [(t, c) for t, c in calls if (now - t).total_seconds() < seconds]
        used = sum(c for _, c in in_window)
        if used + cost > allowed:
            if name == "per_minute":
                oldest = min(t for t, _ in in_window)
                return "wait", max(1.0, seconds - (now - oldest).total_seconds())
            return "defer", (f"API budget {name}: {used:g} used + {cost:g} > {allowed:g} "
                             f"({api_safety_margin:.0%} of {limits[name]})")
    if calls:
        gap = (now - max(t for t, _ in calls)).total_seconds()
        if gap < limits["min_seconds_between_calls"]:
            return "wait", limits["min_seconds_between_calls"] - gap
    return "call", None


def block_until_from_retry_after(retry_after):
    """Retry-After in seconds or as an HTTP date; otherwise the next UTC midnight (daily quotas)."""
    now = utc_now()
    if retry_after:
        value = str(retry_after).strip()
        if value.isdigit():
            return now + timedelta(seconds=int(value))
        try:
            return parsedate_to_datetime(value).astimezone(timezone.utc).replace(tzinfo=None)
        except (TypeError, ValueError):
            pass
    return datetime(now.year, now.month, now.day) + timedelta(days=1)


def active_block(quota_group):
    """The active block of a quota group (dict) or None."""
    if not spark.catalog.tableExists(source_blocks_table):
        return None
    row = (spark.table(source_blocks_table)
           .filter((F.col("quota_group") == quota_group) & F.col("unblocked_at").isNull()
                   & (F.col("blocked_until") > F.lit(utc_now())))
           .orderBy(F.desc("blocked_until")).first())
    return row.asDict() if row else None


def block_quota_group(quota_group, until, reason):
    create_api_governance_tables()
    (spark.createDataFrame([(quota_group, utc_now(), until, str(reason)[:500], None)],
                           "quota_group STRING, blocked_at TIMESTAMP, blocked_until TIMESTAMP, reason STRING, "
                           "unblocked_at TIMESTAMP")
          .write.mode("append").saveAsTable(source_blocks_table))
    print(f"⛔ quota group {quota_group} blocked until {until} UTC: {str(reason)[:150]}")


def unblock_quota_group(quota_group):
    """Admin: lifts the active blocks of a quota group (only when you know the provider allows it again)."""
    create_api_governance_tables()
    spark.sql(f"UPDATE {source_blocks_table} SET unblocked_at = current_timestamp() "
              f"WHERE quota_group = '{quota_group}' AND unblocked_at IS NULL")
    print(f"✅ quota group {quota_group} unblocked")